In [30]:
import pandas as pd
pd.ExcelFile('US_Regional_Sales_Data.xlsx').sheet_names

# .sheet_names para ver los nombres de las hojas del archivo excel para luego crear un df por hoja

['Sales Orders Sheet',
 'Customers Sheet',
 'Store Locations Sheet',
 'Products Sheet',
 'Sales Team Sheet']

In [31]:
Sales_orders=pd.read_excel('US_Regional_Sales_Data.xlsx',sheet_name='Sales Orders Sheet')
Customers=pd.read_excel('US_Regional_Sales_Data.xlsx',sheet_name='Customers Sheet')
Store_location=pd.read_excel('US_Regional_Sales_Data.xlsx',sheet_name='Store Locations Sheet')
Products=pd.read_excel('US_Regional_Sales_Data.xlsx',sheet_name='Products Sheet')
Sales_team=pd.read_excel('US_Regional_Sales_Data.xlsx',sheet_name='Sales Team Sheet')
hojas = [
    'Sales Orders Sheet',
    'Customers Sheet',
    'Store Locations Sheet',
    'Products Sheet',
    'Sales Team Sheet'
]

for hoja in hojas:
    df = pd.read_excel('US_Regional_Sales_Data.xlsx', sheet_name=hoja)
    print(f"\nHoja: {hoja}")
    print(df.columns)

# Se crearon df´s por hoja del archivo 'US_Regional_Sales_Data.xlsx'
# y luego se creo un bucle for para mostrar las columnas de cada df
# y poder identificar los ID correspondientes


Hoja: Sales Orders Sheet
Index(['OrderNumber', 'Sales Channel', 'WarehouseCode', 'ProcuredDate',
       'OrderDate', 'ShipDate', 'DeliveryDate', 'CurrencyCode', '_SalesTeamID',
       '_CustomerID', '_StoreID', '_ProductID', 'Order Quantity',
       'Discount Applied', 'Unit Price', 'Unit Cost'],
      dtype='object')

Hoja: Customers Sheet
Index(['_CustomerID', 'Customer Names'], dtype='object')

Hoja: Store Locations Sheet
Index(['_StoreID', 'City Name', 'County', 'StateCode', 'State', 'Type',
       'Latitude', 'Longitude', 'AreaCode', 'Population', 'Household Income',
       'Median Income', 'Land Area', 'Water Area', 'Time Zone'],
      dtype='object')

Hoja: Products Sheet
Index(['_ProductID', 'Product Name'], dtype='object')

Hoja: Sales Team Sheet
Index(['_SalesTeamID', 'Sales Team', 'Region'], dtype='object')


In [32]:
df1 = Sales_orders.merge(Customers,on='_CustomerID',how='left',validate='many_to_one')
df2=df1.merge(Store_location,on='_StoreID', how='left',validate='many_to_one')
df3=df2.merge(Products,on='_ProductID', how='left',validate='many_to_one')
df_base=df3.merge(Sales_team,on='_SalesTeamID', how='left',validate='many_to_one')
df_base.columns

# Se unen los df´s en 'df_base usando merge y una validación many to one para una correcta union

Index(['OrderNumber', 'Sales Channel', 'WarehouseCode', 'ProcuredDate',
       'OrderDate', 'ShipDate', 'DeliveryDate', 'CurrencyCode', '_SalesTeamID',
       '_CustomerID', '_StoreID', '_ProductID', 'Order Quantity',
       'Discount Applied', 'Unit Price', 'Unit Cost', 'Customer Names',
       'City Name', 'County', 'StateCode', 'State', 'Type', 'Latitude',
       'Longitude', 'AreaCode', 'Population', 'Household Income',
       'Median Income', 'Land Area', 'Water Area', 'Time Zone', 'Product Name',
       'Sales Team', 'Region'],
      dtype='object')

In [33]:
df_base['ProcurementDays']=df_base['OrderDate']-df_base['ProcuredDate']
df_base['ShippingDays']=df_base['ShipDate']-df_base['OrderDate']
df_base['DeliveryDays']=df_base['DeliveryDate']-df_base['ShipDate']
df_base['CustomerDays']=df_base['ShippingDays']+df_base['DeliveryDays']

# Se calculan las diferencias de tiempo entre fechas clave para medir duración de cada etapa
# y el total del proceso.

In [34]:
df_base[['OrderDate','ProcuredDate','ShipDate','DeliveryDate','ProcurementDays','ShippingDays','DeliveryDays','CustomerDays']].dtypes
# Validacion de fechas en formato datetime64 y diferencias en timedelta64 para asegurar calculos correctos.

OrderDate           datetime64[ns]
ProcuredDate        datetime64[ns]
ShipDate            datetime64[ns]
DeliveryDate        datetime64[ns]
ProcurementDays    timedelta64[ns]
ShippingDays       timedelta64[ns]
DeliveryDays       timedelta64[ns]
CustomerDays       timedelta64[ns]
dtype: object

In [35]:
df_base['CustomerDaysInterval']=pd.cut(df_base['CustomerDays'].dt.days,bins=[0,15,30,45,60,75,90])
df_base['CustomerDaysInterval'].head()

# Se convierte CustomerDays a días numéricos con .dt.days y se agrupa en intervalos usando pd.cut.

0    (15, 30]
1    (30, 45]
2    (30, 45]
3     (0, 15]
4    (15, 30]
Name: CustomerDaysInterval, dtype: category
Categories (6, interval[int64, right]): [(0, 15] < (15, 30] < (30, 45] < (45, 60] < (60, 75] < (75, 90]]

In [36]:
tabla_pivote = pd.pivot_table(df_base,index='Sales Team',
                              columns='CustomerDaysInterval',values='OrderNumber',
                              aggfunc='count',
                              margins=True, observed=True
)
tabla_pivote

# Tabla pivote que muestra el conteo de órdenes por equipo de ventas y rangos de días, incluyendo totales.

CustomerDaysInterval,"(0, 15]","(15, 30]","(30, 45]",All
Sales Team,,,,
Adam Hernandez,81,179,42,302
Anthony Berry,76,181,41,298
Anthony Torres,81,162,45,288
Carl Nguyen,96,178,40,314
Carlos Miller,65,148,34,247
Chris Armstrong,92,156,44,292
Donald Reynolds,97,163,36,296
Douglas Tucker,87,145,45,277
Frank Brown,85,150,40,275


El 30 % de las órdenes llega al cliente en 15 días o menos, el 56 % entre 16 y 30 días y el 14 % entre 31 y 45 días. Ninguna supera los 45 días.

In [37]:
df_base['GrossMargin']=df_base['Order Quantity']*(df_base['Unit Price']*(1-df_base['Discount Applied'])-df_base['Unit Cost'])
# Cálculo del Gross Margin como diferencia entre ingresos y costos asociados.

In [38]:
# Se asigna el porcentaje de comisión según el rango de GrossMargin utilizando pd.cut
df_base['CommissionsPercentage']= pd.cut(df_base['GrossMargin'],
                                         bins=[0,100,1000,10000,100000],
                                         labels=[0.05,0.1,0.15,0.2]
                                         ).astype(float)

In [39]:
# Se calcula el monto de la comisión utilizando las columnas creadas previamente
df_base['CommissionsAmount']=df_base['GrossMargin']*df_base['CommissionsPercentage']

In [40]:
# Se calcula el margen neto, restando las comisiones de los equipos de ventas al margen bruto
df_base['Net Margin']=df_base['GrossMargin']-df_base['CommissionsAmount']

In [41]:
tabla_pivote1=pd.pivot_table(df_base,index='Sales Team',
                             values=['GrossMargin','Net Margin','CommissionsAmount'],
                             aggfunc='sum',
                             margins=True).round(2)
tabla_pivote1.sort_values(by='Net Margin')

# Tabla pivote que resume la suma de GrossMargin, Net Margin y CommissionsAmount por equipo de ventas, incluyendo totales.

,CommissionsAmount,GrossMargin,Net Margin
Sales Team,,,
Joshua Bennett,108049.11,661726.54,569242.06
Frank Brown,104839.47,670881.25,570076.79
Carlos Miller,110882.09,668752.93,580326.02
Paul Holmes,114224.89,705564.60,598867.37
Keith Griffin,112940.12,675453.60,599688.29
Shawn Torres,114290.96,705619.61,600950.39
Stephen Payne,116111.48,707156.79,606241.58
Joshua Ryan,114204.91,702157.82,606904.87
Shawn Wallace,116942.05,726737.68,624236.47


Margen: el margen bruto total es de 21,67 M. Las comisiones suman 3,44 M (cerca del 16 % del margen bruto) y el margen neto queda en 18,23 M.
Mejor equipo: Donald Reynolds lidera el margen neto (754 mil), seguido de Shawn Cook (724 mil) y Joshua Little (714 mil).
Brecha entre equipos: Reynolds supera en un 32 % al equipo con menor margen neto, Joshua Bennett (569 mil).
Volumen no es lo mismo que margen: Todd Roberts gestiona más órdenes (340), pero Reynolds genera más margen con menos (296). Por orden, el margen bruto es de cerca de 2.990 contra 2.340.